# Differential gene expression

In [1]:
# Path and system utilities
import os                    # Operating system interface
import sys                   # System-specific parameters and functions
import glob                  # File pattern matching
from pathlib import Path     # Object-oriented filesystem paths
from pyhere import here      # Reproducible project paths
import gc
import time

# Single-cell data handling
import anndata as ad            # Core data structure for single-cell data
import scanpy as sc

# milo
import pertpy as pt
milo = pt.tl.Milo()
import mudata as md
from mudata import MuData

# Parallel processing
from joblib import Parallel, delayed, parallel_backend


# dataframes
import pandas as pd
import numpy as np
from collections import defaultdict
# plotting
import matplotlib.pyplot as plt 

# Diff genes
from flash_mm import lmmfit, lmmtest, contrast_matrix, sslmm, lmm
from statsmodels.stats.multitest import multipletests

# Custom modules and functions
sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import diff_genes as dg
import misc as mi
import dg_milo as dm

/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Paths
base_dir = str(here('data/differential_abundance/'))
plot_dir = os.path.join(base_dir, 'plot') 
files_dir = os.path.join(base_dir, 'files') 
de_dir = os.path.join(base_dir, 'de_analysis_groups') 
de_overall_dir = os.path.join(base_dir, 'de_analysis_overall') 
objects_dir = os.path.join(base_dir, 'objects') 
anndata_dir = str(here('data/anndata/'))

In [3]:
mdata =  md.read(os.path.join(objects_dir, "mdata_666_annotation.h5mu"), backed='r')

## Functions

In [4]:
def get_comparisons(anno_direction_values):
    """Build list of (c1, c2, label) comparisons from anno_direction values."""
    # Only keep up/down clusters
    clusters = [c for c in anno_direction_values.unique()
                if isinstance(c, str) and c.endswith(("_up", "_down"))]

    # Build comparisons: prefer up vs down, fallback to vs other
    groups = defaultdict(dict)
    for c in clusters:
        base, direction = c.rsplit("_", 1)
        groups[base][direction] = c
    
    comparisons = []
    for base, g in groups.items():
        if "up" in g and "down" in g:
            comparisons.append((g["up"], g["down"], f"{base}_up_vs_down"))
        else:
            for c in g.values():
                comparisons.append((c, "other", f"{c}_vs_other"))
    return comparisons

def get_subset(adata, df_sub, comp_label, c1, c2):
    """Subset adata to cells in comparison and attach group labels."""
    labels = df_sub["anno_direction"].map(
        lambda x: x if x in [c1, c2] else ("other" if c2 == "other" else None)
    ).rename(comp_label)

    subset = adata[adata.obs_names.isin(df_sub.index)].to_memory()
    subset.obs = subset.obs.join(labels)
    return subset[subset.obs[comp_label].notna()].copy()

## Differential gene expression

#### Map cells to each neighborhood

In [5]:
adata            = mdata["rna"]
nhoods           = adata.obsm["nhoods"]                          
ref_cells        = adata.obs_names[adata.obs["nhood_ixs_refined"] == 1]

rows, cols = nhoods.nonzero()
df = pd.DataFrame({
    "cell"       : adata.obs_names[rows],
    "nhood_id"   : cols,
    "index_cell" : ref_cells[cols],
})

#### Combine differential abundance and cell data

In [6]:
# ----------------------------- SET UP -----------------------------------
print("Setting up variables")
prefix           = "t2d_vs_nd"
out_dir          = de_overall_dir 
da_results_path  = os.path.join(files_dir, f"{prefix}.csv")

nhood_anno_key   = "nhood_annotation"
sample_key       = "ic_id_platform_adjusted_sample"
donor_key        = "ic_id_donor_overall"
disease_key      = "disease_harmonized"

# ----------------------------- LOAD -------------------------------------
da = pd.read_csv(da_results_path, index_col=0)
# ----------------------------- PREPROCESS -------------------------------
print("Combine differential abundance data with per cell neighborhoods")
df_t2d = df.merge(da, how="left")
df_t2d ["direction"] = "no_change"

# Direction of regulations
df_t2d.loc[(df_t2d["SpatialFDR"] <= 0.1) & (df_t2d["logFC"] > 0), "direction"] = "up"
df_t2d.loc[(df_t2d["SpatialFDR"] <= 0.1) & (df_t2d["logFC"] < 0), "direction"] = "down"
df_t2d ["anno_direction"] = df_t2d[nhood_anno_key] + "_" + df_t2d["direction"]

df_t2d = df_t2d .set_index('cell')

Setting up variables
Combine differential abundance data with per cell neighborhoods


#### Differential expression analysis on subset of data

In [ ]:
# Store results per contrast
results = {}
for anno_id in list(df_t2d['nhood_annotation'].drop_duplicates()):
    t0_anno = time.time()
    
    print(f"\nProcessing: {anno_id}")

    # Subset to cell type (annotated neighborhoods)
    df_sub = (df_t2d.loc[df_t2d[nhood_anno_key] == anno_id, ["anno_direction"]]
               .loc[lambda x: ~x.index.duplicated(keep="first")])

    # Define comparisons
    comparisons = get_comparisons(df_sub["anno_direction"])
    
    if not comparisons:
        print(f"  No DA neighborhoods, skipping")
        continue
        
    # Diff for comparison
    for c1, c2, comp_label in comparisons:
        print(f"  DE: {comp_label}")
        
        # Get cells to be used for comparison
        subset = get_subset(adata, df_sub, comp_label, c1, c2)

        # Define fixed and random effect
        fixed_effect = comp_label
        random_effect = donor_key

        # Get gene expression (log-transformed counts)(Genes x Cells)
        Y = subset.X.T 
        Y_names = subset.var_names.tolist()

        # Ensure Y is a dense numpy array if it's currently sparse
        if hasattr(Y, "toarray"):
            Y = Y.toarray()
            
        # Filter genes - remove genes that are not expressed in atleast 5 % of cells
        n_cells = Y.shape[1]
        min_cells = 0.05 * n_cells
        expressed_mask = np.asarray((Y > 0).sum(axis=1)).flatten() >= min_cells
        Y = Y[expressed_mask, :]
        Y_names = np.array(Y_names)[expressed_mask].tolist()
        
        # Filter genes - remove genes with small variance
        # Calculate variance for each gene (rows)
        gene_vars = np.var(Y, axis=1)
        
        # Keep only genes with a variance above a tiny threshold
        mask = gene_vars > 1e-6
        Y = Y[mask, :]
        Y_names = np.array(Y_names)[mask].tolist()
        
        print(f"Remaining genes after variance filtering: {len(Y_names)}")
        
        # Get observations 
        obs = subset.obs.copy()

        # Remove subset from memory
        del subset
        gc.collect()
        
        # X: Design matrix for fixed effects
        obs[fixed_effect] = obs[fixed_effect].astype('category')
        trt_labels = obs[fixed_effect].values
        
        # Get the unique categories to ensure consistent ordering
        categories = obs[fixed_effect].cat.categories.tolist()
        
        # Create the model matrix X (One-Hot Encoding)
        # This creates a column for each category (e.g., A and B)
        X = np.column_stack([trt_labels == cat for cat in categories]).astype(float)
        
        # Create the names for your columns
        X_names = [cat for cat in categories]
        
        print(f"Model Matrix Shape: {X.shape}")
        print(f"Columns: {X_names}")
        
        # Z: design matrix for random effects
        # Identify the column containing your random effect groups (e.g., donor ids)
        # Ensure it is categorical to maintain a stable order
        samples = obs[random_effect].astype('category')
        sample_names = samples.cat.categories.tolist()
        
        # Create the Z matrix (Random Effects Design Matrix)
        # Each column represents one donor; 1.0 if the cell belongs to that sample, else 0.0
        Z = np.column_stack([samples == s for s in sample_names]).astype(float)
        
        # Store names for your records (useful for downstream variance analysis)
        Z_names = [s for s in sample_names]
        
        d = Z.shape[1]
        
        print(f"Z matrix shape: {Z.shape}") # Should be (n_cells, n_samples)
        
        # Ensure X and Z are standard numpy arrays (N_cells x N_features)
        X = np.asarray(X, dtype=np.float64)
        Z = np.asarray(Z, dtype=np.float64)
        
        # Verify dimensions before fitting
        print(f"Y shape (Genes, Cells): {Y.shape}")
        print(f"X shape (Cells, Fixed): {X.shape}")
        print(f"Z shape (Cells, Random): {Z.shape}")
        
        # FLASH-MM using summary statistics
        method = "REML" 
        model_vars = X_names  
        
        # Define contrasts (both directions)
        c1, c2 = X_names[0], X_names[1]
        
        contrast = {
            f"{c1}_vs_{c2}": f"{c1}-{c2}",
            f"{c2}_vs_{c1}": f"{c2}-{c1}"
        }
        
        cm = contrast_matrix(contrast, model_vars)
        
        # Fit Model
        ss = sslmm(X, Y, Z)
        fit = lmm(
            XX=ss["XX"], 
            XY=ss["XY"], 
            ZX=ss["ZX"], 
            ZY=ss["ZY"], 
            ZZ=ss["ZZ"], 
            Ynorm=ss["Ynorm"], 
            n=ss["n"], 
            d=[Z.shape[1]], 
            method=method, 
            Y_names=Y_names, 
            X_names=X_names,
            output_cov=True, 
            output_RE=False
        )
        
        # Run the Statistical Test
        test_output = lmmtest(fit, contrast=cm)
        
        for cname in contrast.keys():
            coef_col = f"{cname}_coef"
            t_stat_col = f"{cname}_t"
            p_val_col = f"{cname}_p"
            
            df_dea = pd.DataFrame({
                'logFC': test_output[coef_col].values,
                't_stat': test_output[t_stat_col].values,
                'p_value': test_output[p_val_col].values
            }, index=Y_names)
            
            # Multiple Testing Correction (FDR)
            mask = ~df_dea['p_value'].isna()
            df_dea['adj_p_val'] = np.nan
            if mask.any():
                df_dea.loc[mask, 'adj_p_val'] = multipletests(
                    df_dea.loc[mask, 'p_value'], method='fdr_bh'
                )[1]
            
            df_dea = df_dea.sort_values('adj_p_val')
            
            print(f"Top Differential Genes for {cname}:")
            print(df_dea.head(10))
            
            results[cname] = df_dea
        print(f"Finished {anno_id} in {time.time() - t0_anno:.2f} sec")
        
# Save
df_all = pd.concat(results, names=["contrast", "gene_symbol"])
df_all = df_all.reset_index(level="contrast").reset_index()

df_all.to_csv(os.path.join(de_overall_dir, f"{prefix}_diff_genes_all_donors.csv"), index = False)


Processing: alpha
  DE: alpha_up_vs_down
Remaining genes after variance filtering: 10494
Model Matrix Shape: (6346, 2)
Columns: ['alpha_down', 'alpha_up']
Z matrix shape: (6346, 228)
Y shape (Genes, Cells): (10494, 6346)
X shape (Cells, Fixed): (6346, 2)
Z shape (Cells, Random): (6346, 228)
Top Differential Genes for alpha_down_vs_alpha_up:
           logFC     t_stat        p_value      adj_p_val
MT1E   -0.798325 -39.115839  5.149963e-300  5.404371e-296
MT1X   -0.655438 -34.534890  1.217328e-239  6.387320e-236
MT2A   -0.857641 -33.286259  4.627613e-224  1.618739e-220
INS    -0.841607 -28.003446  8.407009e-163  2.205579e-159
PDK4    0.916255  22.972298  3.022134e-112  6.342855e-109
MT1F   -0.337603 -22.637224  3.575352e-109  6.253290e-106
VIM     0.905462  21.226632   1.167352e-96   1.750028e-93
IFITM3 -0.287381 -20.077041   5.629404e-87   7.384370e-84
SPP1   -0.378515 -19.718782   4.697049e-84   5.476759e-81
MT1G   -0.287786 -19.586504   5.479867e-83   5.750573e-80
Top Differential G

/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/flash_mm/lmm.py:268: RuntimeWarning: invalid value encountered in sqrt
  sebeta.append(np.sqrt(np.diag(covb)))
/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/flash_mm/lmmtest.py:76: RuntimeWarning: invalid value encountered in sqrt
  se = np.sqrt(np.diag(contrast.T @ cov_j @ contrast))


Top Differential Genes for myeloid_down_vs_myeloid_up:
           logFC     t_stat       p_value     adj_p_val
DUSP26  4.737815  97.323214  0.000000e+00  0.000000e+00
MEG3   -1.126396 -13.696170  4.875543e-40  2.504810e-36
RASD1  -0.638482 -12.766286  2.725863e-35  9.336081e-32
SCG3   -1.156357 -11.967732  2.024321e-31  5.199974e-28
TFF3   -0.695649 -11.467550  4.263563e-29  8.761622e-26
SCG5   -1.824539 -11.380696  1.059538e-28  1.814459e-25
CPE    -1.606416 -10.925313  1.142745e-26  1.677386e-23
SEZ6L2 -0.619078 -10.515493  6.746637e-25  7.702411e-22
DLK1   -0.580295 -10.517295  6.628596e-25  7.702411e-22
UCHL1  -0.821230 -10.476625  9.866352e-25  1.013768e-21
Top Differential Genes for myeloid_up_vs_myeloid_down:
           logFC     t_stat       p_value     adj_p_val
DUSP26 -4.737815 -97.323214  0.000000e+00  0.000000e+00
MEG3    1.126396  13.696170  4.875543e-40  2.504810e-36
RASD1   0.638482  12.766286  2.725863e-35  9.336081e-32
SCG3    1.156357  11.967732  2.024321e-31  5.19997

/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/flash_mm/lmm.py:268: RuntimeWarning: invalid value encountered in sqrt
  sebeta.append(np.sqrt(np.diag(covb)))
/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/flash_mm/lmmtest.py:76: RuntimeWarning: invalid value encountered in sqrt
  se = np.sqrt(np.diag(contrast.T @ cov_j @ contrast))


Top Differential Genes for stellate_activated_down_vs_stellate_activated_up:
           logFC     t_stat       p_value     adj_p_val
CALD1   1.689741  15.662616  1.250156e-52  1.351544e-48
LUCAT1 -0.387762 -13.634667  8.983374e-41  4.855963e-37
BGN     1.575605  13.569693  2.045205e-40  7.370236e-37
CXCL8  -0.942466 -13.545322  2.782164e-40  7.519495e-37
IGFBP7  1.579080  13.509538  4.367694e-40  9.443828e-37
COL1A2  1.737662  13.243350  1.212005e-38  2.183832e-35
COL3A1  1.628851  12.932158  5.493199e-37  8.483854e-34
COL1A1  1.704294  12.357625  5.117563e-34  6.915746e-31
PDGFRB  1.196006  12.342742  6.087440e-34  7.312368e-31
KCNK1  -0.327831 -12.294582  1.066092e-33  1.152552e-30
Top Differential Genes for stellate_activated_up_vs_stellate_activated_down:
           logFC     t_stat       p_value     adj_p_val
CALD1  -1.689741 -15.662616  1.250156e-52  1.351544e-48
LUCAT1  0.387762  13.634667  8.983374e-41  4.855963e-37
BGN    -1.575605 -13.569693  2.045205e-40  7.370236e-37
CXCL8 

/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/flash_mm/lmm.py:268: RuntimeWarning: invalid value encountered in sqrt
  sebeta.append(np.sqrt(np.diag(covb)))


Top Differential Genes for endmt_early_down_vs_other:
            logFC      t_stat       p_value     adj_p_val
BOLA2B  -1.507671  -72.792763  0.000000e+00  0.000000e+00
MRPL53  -1.656406 -125.150289  0.000000e+00  0.000000e+00
SLCO2A1  0.733247   19.562885  4.661097e-82  1.571566e-78
NKX2-3   0.793091   15.829196  4.730322e-55  1.196180e-51
CD320    0.986865   14.190639  8.337842e-45  1.686745e-41
KDR      0.949748   13.642750  1.319099e-41  2.223781e-38
SMAD6    0.466696   13.409977  2.783583e-40  4.022277e-37
ATOH8    0.320717   12.536286  1.691055e-35  2.138128e-32
ID1      1.079537   12.289450  3.354952e-34  3.770594e-31
SGK1     0.740558   12.037246  6.710211e-33  6.787378e-30
Top Differential Genes for other_vs_endmt_early_down:
            logFC      t_stat       p_value     adj_p_val
BOLA2B   1.507671   72.792763  0.000000e+00  0.000000e+00
MRPL53   1.656406  125.150289  0.000000e+00  0.000000e+00
SLCO2A1 -0.733247  -19.562885  4.661097e-82  1.571566e-78
NKX2-3  -0.793091  -15

/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/flash_mm/lmm.py:268: RuntimeWarning: invalid value encountered in sqrt
  sebeta.append(np.sqrt(np.diag(covb)))
/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/flash_mm/lmm.py:267: RuntimeWarning: invalid value encountered in log
  loglike.append(-(n - pres) * (1 + np.log(2 * np.pi * s[k])) / 2 + logdet / 2)
/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/flash_mm/lmmtest.py:76: RuntimeWarning: invalid value encountered in sqrt
  se = np.sqrt(np.diag(contrast.T @ cov_j @ contrast))


Top Differential Genes for mixed_down_vs_mixed_up:
             logFC     t_stat        p_value      adj_p_val
COL4A1    2.389937  60.803673   0.000000e+00   0.000000e+00
COL4A2    2.041307  52.151329   0.000000e+00   0.000000e+00
PLVAP     2.607523  65.107817   0.000000e+00   0.000000e+00
PPP1R16B  0.301990  87.826327   0.000000e+00   0.000000e+00
GNG11     2.115760  52.992573   0.000000e+00   0.000000e+00
ENG       2.129401  49.263909  2.819120e-312  4.995950e-309
RGCC      2.184576  46.593603  1.005216e-290  1.526923e-287
SPARC     1.788294  43.816547  4.741888e-268  6.302562e-265
FLT1      1.949951  43.619969  1.947091e-266  2.300380e-263
STC1      2.103001  42.518722  2.214652e-257  2.354840e-254
Top Differential Genes for mixed_up_vs_mixed_down:
             logFC     t_stat        p_value      adj_p_val
COL4A1   -2.389937 -60.803673   0.000000e+00   0.000000e+00
COL4A2   -2.041307 -52.151329   0.000000e+00   0.000000e+00
PLVAP    -2.607523 -65.107817   0.000000e+00   0.000000e+0